# IMEN266 · Ch.4 · Class Problems 4 — Transient analysis

**How this notebook works** — every problem follows the same 5 steps:

> **Problem → Model → Analytic → Simulate & Visualize → Experiment**

The analytic answer and the simulation must agree. *That agreement — not the
instructor, not an AI — is your ground truth.*

**In-class version only:** cells marked `# TODO` are for you to complete during
class. A `check(...)` cell right after tells you immediately whether your answer
matches the reference value.

▶ Open in Colab: `https://colab.research.google.com/github/youngmko/imen266-2026/blob/main/ch04/notebooks/CP4_transient_analysis.ipynb`

**The two formulas of this notebook** *(slide p. 23/52; Companion Proofs 1–2)*

- initial distribution $a=[P(X_0=0),P(X_0=1),\dots]$, and $p_n=[P(X_n=0),P(X_n=1),\dots]$;
- $p_1=aP,\;p_2=p_1P=aP^2,\;\dots,\;\boxed{p_n=aP^n}$ (Chapman–Kolmogorov: $P^{(m+n)}=P^{(m)}P^{(n)}$);
- path probabilities multiply along the path:
  $P(X_0=i_0,X_1=i_1,\dots,X_n=i_n)=a_{i_0}\,P_{i_0i_1}P_{i_1i_2}\cdots P_{i_{n-1}i_n}$.

In code: `chain.n_step(n)` is $P^n$, `chain.distribution(a, n)` is $aP^n$,
`chain.path_probability([...], a)` is the product above.

In [ ]:
# --- IMEN266 setup (Ch.4): course package + helpers ---------------------------
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

try:                                   # local clone / PLMS zip: package is on the path
    from imen266.dtmc import DTMC
except ImportError:                    # Colab: fetch the course package from GitHub (~5 s)
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/youngmko/imen266-2026.git"], check=True)
    from imen266.dtmc import DTMC
from imen266.check import check        # [TODO]/[OK]/[X] self-check (unfinished TODOs never crash)

rng = np.random.default_rng(2026)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True,
                     "grid.alpha": .3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11,
                     "legend.frameon": False})

def build_chain(P, states=None):
    """DTMC(P) if P is complete and valid; otherwise None (keeps 'Run all' alive)."""
    P = np.asarray(P, dtype=float)
    if np.isnan(P).any():
        print("[TODO] P still contains NaN entries - finish the TODO above."); return None
    try:
        return DTMC(P, states)
    except ValueError as e:
        print("[X  ] P is not a valid transition matrix:", e); return None

def check_rows(P):
    """Every row of a transition matrix must sum to 1 (and be nonnegative)."""
    P = np.asarray(P, dtype=float)
    err = None if np.isnan(P).any() else float(np.abs(P.sum(axis=1) - 1).max() + (P < 0).sum())
    return check("rows sum to 1, entries >= 0", err, 0.0, tol=1e-9)

try:
    from ipywidgets import interact, FloatSlider, IntSlider, FloatLogSlider
    HAS_W = True
except Exception:
    HAS_W = False

def show(fn, **sliders):
    """interact() if widgets are available; otherwise draw once with defaults."""
    if HAS_W:
        interact(fn, **sliders)
    else:
        fn(**{k: getattr(v, "value", v) for k, v in sliders.items()})

print("Setup OK.  DTMC imported.  Widgets available:", HAS_W)

---
## Problem 1 — Pohang weather, a few days ahead *(slide p. 20/52)*

> Consider the 3-state DTMC for weather in the city of Pohang in the previous
> class problem. Answer the following questions (all independent):
> (a) According to weather forecast today will be rainy with probability 0.3 and
> sunny with probability 0.4. What is the probability it would be cloudy
> tomorrow? How about day after? What about 3 days from today?
> (b) You know that today is sunny. What is the probability that it would be
> rainy day after tomorrow?
> (c) If yesterday was sunny, day before was sunny and today is rainy, what is the
> probability that it would be cloudy day after tomorrow?

### Model
States $0$ sunny, $1$ cloudy, $2$ rainy with $P$ from CP3 #2. (a) $a=(0.4,\,0.3,\,0.3)$
(cloudy gets the rest), answer $=(aP^n)_1$ for $n=1,2,3$. (b) $(P^2)_{02}$. (c) By the
Markov property the two earlier days are irrelevant: $(P^2)_{21}$.

In [ ]:
W = [0, 1, 2]                          # X_n = 0 sunny, 1 cloudy, 2 rainy (as on the slides)
P_w = np.array([[0.5, 0.3, 0.2], [0.5, 0.2, 0.3], [0.4, 0.5, 0.1]])
weather = DTMC(P_w, W)

# TODO 1a: initial distribution from the forecast (order 0 sunny, 1 cloudy, 2 rainy)
a = np.array([np.nan, np.nan, np.nan])
p1, p2, p3 = (weather.distribution(a, n) for n in (1, 2, 3))
ans_a = (p1[1], p2[1], p3[1])                      # cloudy tomorrow / day after / 3 days

# TODO 1b/1c: pick the right entry of the right power of P  (indices: S=0, C=1, R=2)
ans_b = ...                                        # P(rainy day after tomorrow | sunny today)
ans_c = ...                                        # P(cloudy day after tomorrow | rainy today, and the past)

print("(a)", np.round(ans_a, 4)); print("(b)", ans_b); print("(c)", ans_c)

In [ ]:
check("(a) cloudy tomorrow",  ans_a[0], 0.33,  tol=1e-9)
check("(a) cloudy day after", ans_a[1], 0.307, tol=1e-9)
check("(b) P^2[S,R]",         ans_b, 0.21, tol=1e-9)
check("(c) P^2[R,C]",         ans_c, 0.27, tol=1e-9)

In [ ]:
# --- Simulate 200,000 three-day forecasts drawn from a, and (b)/(c) episodes -----
if np.isnan(a).any():
    print("(finish TODO 1a first)")
else:
    N = 200_000
    x0 = rng.choice(3, size=N, p=a)
    # vectorised path sampling: cumulative rows + uniform draws
    C = P_w.cumsum(axis=1)
    def step(x):
        return (rng.random(len(x))[:, None] > C[x]).sum(axis=1)
    x1 = step(x0); x2 = step(x1); x3 = step(x2)
    print("(a) sim P(cloudy) tomorrow/day after/3 days:",
          np.round([np.mean(x1 == 1), np.mean(x2 == 1), np.mean(x3 == 1)], 4))
    print(f"(b) sim P(R day after | S today) = {np.mean(x2[x0 == 0] == 2):.4f}")
    print(f"(c) sim P(C day after | R today) = {np.mean(x2[x0 == 2] == 1):.4f}")

    path = weather.distribution_path(a, 12)
    plt.figure()
    for j, s in enumerate(["0 sunny", "1 cloudy", "2 rainy"]): plt.plot(path[:, j], "o-", label=s)
    plt.xlabel("days from today"); plt.ylabel("P(weather)"); plt.legend()
    plt.title("The forecast forgets its starting point within ~4 days  (p_n = a P^n)"); plt.show()

In [ ]:
def forecast_exp(p_sunny=0.4, p_rainy=0.3, n=3):
    if p_sunny + p_rainy > 1: print("probabilities exceed 1"); return
    a0 = np.array([p_sunny, 1 - p_sunny - p_rainy, p_rainy])
    path = weather.distribution_path(a0, n)
    plt.figure(); w = .25
    for j, s in enumerate(["0 sunny", "1 cloudy", "2 rainy"]): plt.bar(np.arange(n + 1) + (j-1)*w, path[:, j], w, label=s)
    plt.xticks(range(n + 1)); plt.xlabel("day"); plt.ylabel("probability"); plt.legend()
    plt.title(f"p_n = a P^n for a = {np.round(a0, 2)}"); plt.show()

show(forecast_exp,
     p_sunny=FloatSlider(min=0, max=1, step=.05, value=.4) if HAS_W else .4,
     p_rainy=FloatSlider(min=0, max=1, step=.05, value=.3) if HAS_W else .3,
     n=IntSlider(min=1, max=10, value=3) if HAS_W else 3)

**Think about it** — (i) In (c), which axiom/definition lets you throw away
"yesterday was sunny, the day before was sunny"? Write the equality. (ii) The
curves flatten to the same values whatever $a$ you choose in the experiment.
Name those values (CP5 #2 computes them) and say what property of $P$ makes the
curves converge at all.

---
## Problem 2 — Himart, weeks ahead *(slide p. 27/52)*

> Consider the 4-state DTMC for the Himart inventory PCs problem in the previous
> class problems. Assume that the business starts with 5 Paint-yum PCs in hand at
> the beginning. Answer the following questions (all independent):
> (a) What is the initial distribution $a$?
> (b) What is the probability that at the end of that week Paint-yum PCs have to
> be ordered for the next week?
> (c) What is the probability that after 4 weeks there are 3 Paint-yum PCs?
> (d) If you are given that at the beginning of the 17th week there are 3
> Paint-yum PCs, what is the probability that at the beginning of the 20th week
> there would be 5?

### Model
States $(2,3,4,5)$ = Monday stock, $P$ from CP3 #3. (a) $a=(0,0,0,1)$.
(b) An order is triggered when Friday stock $<2$, i.e. $D\ge4$: $P(D\ge4)$.
(Careful: $P_{55}$ also contains the *no-order* case $D=0$.)
(c) $(aP^4)_3$. (d) Time-homogeneity: $(P^{3})_{35}$ — only the gap of 3 weeks matters.

In [ ]:
S_inv = [2, 3, 4, 5]
P_inv = np.zeros((4, 4))
for k, i in enumerate(S_inv):
    for d in range(0, i - 1):
        P_inv[k, S_inv.index(i - d)] += stats.poisson.pmf(d, 3)
    P_inv[k, 3] += stats.poisson.sf(i - 2, 3)
himart = DTMC(P_inv, S_inv)

# TODO 2a: initial distribution over (2, 3, 4, 5) when the store starts with 5
a_inv = np.array([np.nan, np.nan, np.nan, np.nan])
# TODO 2b: P(order at the end of week 1) -- which demand values trigger an order from 5?
ans_b = ...
# TODO 2c/2d: use himart.distribution / himart.n_step  (remember: index of state 3 is 1, of 5 is 3)
ans_c = ...
ans_d = ...
print("(b)", ans_b, "(c)", ans_c, "(d)", ans_d)

In [ ]:
check("(a) a[5] = 1", None if np.isnan(a_inv).any() else a_inv[3], 1.0, tol=1e-9)
check("(b) P(D >= 4)", ans_b, 1 - stats.poisson.cdf(3, 3), tol=1e-9)              # 0.3528
check("(c) (a P^4)_3", ans_c, float((np.array([0, 0, 0, 1.]) @ np.linalg.matrix_power(P_inv, 4))[1]), tol=1e-9)
check("(d) (P^3)_{3,5}", ans_d, float(np.linalg.matrix_power(P_inv, 3)[1, 3]), tol=1e-9)

In [ ]:
# --- Simulate the store 100,000 times for 20 weeks, from a full shelf ------------
if np.isnan(a_inv).any():
    print("(finish TODO 2a first)")
else:
    R, T = 100_000, 20
    monday = np.full((R, T + 1), 5); order = np.zeros((R, T), bool)
    for n in range(T):
        D = rng.poisson(3, R)
        friday = np.maximum(monday[:, n] - D, 0)
        order[:, n] = friday < 2
        monday[:, n + 1] = np.where(order[:, n], 5, friday)
    print(f"(b) sim P(order week 1)     = {order[:, 0].mean():.4f}")
    print(f"(c) sim P(X_4 = 3)          = {np.mean(monday[:, 4] == 3):.4f}")
    sel = monday[:, 17] == 3
    print(f"(d) sim P(X_20 = 5 | X_17 = 3) = {np.mean(monday[sel, 20] == 5):.4f}   (from {sel.sum():,} runs with X_17 = 3)")

    path = himart.distribution_path(a_inv, 12)
    plt.figure(); plt.stackplot(range(13), path.T, labels=[f"stock {s}" for s in S_inv], alpha=.8)
    plt.xlabel("week"); plt.ylabel("P(Monday stock)"); plt.legend(loc="center right")
    plt.title("From a full shelf, the stock distribution settles within ~3 weeks"); plt.show()

In [ ]:
def himart_exp(start=5, n=4):
    a0 = np.zeros(4); a0[S_inv.index(start)] = 1
    pn = himart.distribution(a0, n)
    plt.figure(); plt.bar([str(s) for s in S_inv], pn); plt.ylim(0, 1)
    plt.xlabel("Monday stock"); plt.title(f"P(X_{n} = .) starting from {start}   [P(order this week) = {pn @ P_inv[:, 3] - pn[3]*stats.poisson.pmf(0,3):.3f}]"); plt.show()

show(himart_exp,
     start=IntSlider(min=2, max=5, value=5) if HAS_W else 5,
     n=IntSlider(min=0, max=12, value=4) if HAS_W else 4)

**Think about it** — (i) Why is (b) *not* $P_{55}$? Which two events does
$P_{55}$ lump together? (ii) In (d), "17th week" and "20th week" never entered
the computation. Which property of the chain makes only the difference
$20-17=3$ matter?

---
## Problem 3 — A 4-state chain with a given $a$ and $P$ *(slide p. 24/52)*

> Consider a DTMC with initial distribution $a$ given by $a=(0.3,\,0.5,\,0.1,\,0.1)$
> and transition probability matrix $P$ given by
> $$P=\begin{pmatrix}0.25&0.25&0.25&0.25\\0.1&0.2&0.3&0.4\\0.6&0.1&0.1&0.2\\0.2&0.2&0.2&0.4\end{pmatrix}.$$
> Compute the following:
> (a) $P[X_3=3,X_2=3,X_1=2,X_0=1]$ (b) $P[X_3=3,X_2=2,X_1=1]$
> (c) $P[X_5=2\mid X_1=3]$ (d) $P[X_3=1]$ (e) Distribution of $X_5$
> (f) $E[X_4]$ (g) $\operatorname{Var}[X_4]$

### Model
States $\{1,2,3,4\}$ (index $=$ state $-1$). (a) $a_1P_{12}P_{23}P_{33}$.
(b) marginalise $X_0$: $(aP)_1\,P_{12}\,P_{23}$. (c) $(P^4)_{32}$ (four steps from
time 1 to 5). (d) $(aP^3)_1$. (e) $aP^5$. (f),(g): moments of the pmf $p_4=aP^4$
over the *values* $1,2,3,4$.

In [ ]:
S4 = [1, 2, 3, 4]
a4 = np.array([0.3, 0.5, 0.1, 0.1])
P4 = np.array([[0.25, 0.25, 0.25, 0.25],
               [0.10, 0.20, 0.30, 0.40],
               [0.60, 0.10, 0.10, 0.20],
               [0.20, 0.20, 0.20, 0.40]])
ch4 = DTMC(P4, S4)
vals = np.array(S4)

ans = {}
# TODO 3a: product along the path 1 -> 2 -> 3 -> 3 starting from a  (ch4.path_probability or by hand)
ans["a"] = ...
# TODO 3b: the path 1 -> 2 -> 3 for times 1,2,3 -- X_0 is unknown, so start from (aP)_1
ans["b"] = ...
# TODO 3c: four steps from state 3 to state 2  (mind the 0-based index!)
ans["c"] = ...
ans["d"] = ch4.distribution(a4, 3)[0]                         # (aP^3)_1  (given)
ans["e"] = ch4.distribution(a4, 5)                            # aP^5      (given)
p4 = ch4.distribution(a4, 4)
# TODO 3f/3g: mean and variance of X_4 whose pmf over the VALUES 1,2,3,4 is p4
ans["f"] = ...
ans["g"] = ...
for k, v in ans.items(): print(f"({k})", v if v is Ellipsis else np.round(v, 5))

In [ ]:
check("(a)", ans["a"], 0.3*0.25*0.3*0.1, tol=1e-12)
check("(b)", ans["b"], float((a4 @ P4)[0]) * 0.25 * 0.3, tol=1e-12)
check("(c)", ans["c"], float(np.linalg.matrix_power(P4, 4)[2, 1]), tol=1e-12)
check("(f) E[X4]",   ans["f"], float((a4 @ np.linalg.matrix_power(P4, 4)) @ np.arange(1, 5)), tol=1e-12)
check("(g) Var[X4]", ans["g"], float((a4 @ np.linalg.matrix_power(P4, 4)) @ np.arange(1, 5)**2
                                     - ((a4 @ np.linalg.matrix_power(P4, 4)) @ np.arange(1, 5))**2), tol=1e-12)

In [ ]:
# --- Simulate 300,000 paths of length 5 from a, then read every answer off them --
N = 300_000
C = P4.cumsum(axis=1)
X = np.empty((N, 6), int)
X[:, 0] = rng.choice(4, size=N, p=a4)
for t in range(5):
    X[:, t + 1] = (rng.random(N)[:, None] > C[X[:, t]]).sum(axis=1)
X1 = X + 1                                                    # values 1..4
print(f"(a) sim {np.mean((X1[:,0]==1)&(X1[:,1]==2)&(X1[:,2]==3)&(X1[:,3]==3)):.5f}")
print(f"(b) sim {np.mean((X1[:,1]==1)&(X1[:,2]==2)&(X1[:,3]==3)):.5f}")
print(f"(c) sim {np.mean(X1[X1[:,1]==3, 5]==2):.4f}")
print(f"(d) sim {np.mean(X1[:,3]==1):.4f}")
print( "(e) sim", np.round(np.bincount(X1[:,5], minlength=5)[1:]/N, 4))
print(f"(f) sim E[X4] = {X1[:,4].mean():.4f}     (g) sim Var[X4] = {X1[:,4].var():.4f}")

plt.figure(); w = .35
plt.bar(vals - w/2, np.bincount(X1[:, 5], minlength=5)[1:]/N, w, label="simulated X_5")
plt.bar(vals + w/2, ch4.distribution(a4, 5), w, label="a P^5")
plt.xticks(vals); plt.xlabel("state"); plt.legend(); plt.title("(e): matrix power = Monte-Carlo"); plt.show()

In [ ]:
def moments_exp(n=4):
    path = ch4.distribution_path(a4, 12)
    means = path @ vals; var = path @ vals**2 - means**2
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    ax[0].plot(means, "o-"); ax[0].axvline(n, c="gray", ls="--"); ax[0].set_title(f"E[X_n]   (E[X_{n}] = {means[n]:.4f})"); ax[0].set_xlabel("n")
    ax[1].plot(var, "s-", c="#c0392b"); ax[1].axvline(n, c="gray", ls="--"); ax[1].set_title(f"Var[X_n]   (Var[X_{n}] = {var[n]:.4f})"); ax[1].set_xlabel("n")
    plt.tight_layout(); plt.show()

show(moments_exp, n=IntSlider(min=0, max=12, value=4) if HAS_W else 4)

**Think about it** — (i) (a) and (b) differ by one factor: which one, and why
does (b) need a sum over $X_0$ that (a) does not? (ii) $E[X_n]$ and
$\operatorname{Var}[X_n]$ settle to constants as $n$ grows (CP5 explains what
they converge to). Explain now why the *variance of the state* is not zero in
the limit even though $p_n$ stops changing.

---
## Wrap-up

| # | question type | tool | key idea |
|---|---|---|---|
| 1 | distribution $n$ days ahead | $aP^n$ | forecasts forget $a$ |
| 2 | conditional, time gap | $(P^k)_{ij}$ | only the gap matters (homogeneity) |
| 3 | path / marginal / moments | products, $aP^n$, LOTUS | Markov property = multiply along the path |

**Next steps** → Companion Ch.4 Proofs 1–2 (path probabilities, Chapman–Kolmogorov)
→ CP5 (why $p_n$ converges, and to what).